# Part 3: Topics

An entity graph shows how specific things are connected. **Topics** group related passages so you can explore a broader area of the documents.

The topics in this lab were generated ahead of time by clustering the passage embeddings into a two-level hierarchy. Each topic has a **representative** with a **centroid** (the average embedding of its passages) and a list of **member** passages.

**You'll:**
- Check how much of the document set the topics cover.
- Browse the topic hierarchy.
- Pick a passage and find its nearest neighbors in two ways: directly, and through its nearest topic.

> **Tip:** Open the **Topics** tab in the Cocoarynth graph explorer app to browse the topic hierarchy and its member passages visually.

## Step 1: Load your settings and connect to the topic indexes

The topics are stored in three indexes:

- A **topics** index with the topic representatives, their memberships, and a manifest.
- A **topic vectors** index with each topic's centroid embedding.
- A **topic sources** index with the original passages and their embeddings.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_SEARCH_API_VERSION = "2026-11-01-preview"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

In [ ]:
import json
import textwrap
import time

from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery

TOPICS_INDEX = "grz-cocoarynth-topics-bb1deb018565"
TOPIC_VECTORS_INDEX = "grz-cocoarynth-topic-vectors-bb1deb018565"
SOURCES_INDEX = "grz-cocoarynth-input-f7d06ba6-517dba815a29"

topics_client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, TOPICS_INDEX, search_credential, api_version=AZURE_SEARCH_API_VERSION)
vectors_client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, TOPIC_VECTORS_INDEX, search_credential, api_version=AZURE_SEARCH_API_VERSION)
sources_client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, SOURCES_INDEX, search_credential, api_version=AZURE_SEARCH_API_VERSION)

for index_name, client in ((TOPICS_INDEX, topics_client), (TOPIC_VECTORS_INDEX, vectors_client), (SOURCES_INDEX, sources_client)):
    print(f"{index_name}: {client.get_document_count()} documents")

## Step 2: Check coverage

The manifest records how the topics were built. Not every passage was assigned to a topic: 97 of 106 source chunks were assigned, and 9 were left out. Keep that in mind when you browse: a topic doesn't cover everything in the corpus.

In [ ]:
manifest_row = next(iter(topics_client.search(search_text="*", filter="kind eq 'manifest'", select=["payload"], top=1)))
manifest = json.loads(manifest_row["payload"])
GENERATION = manifest["generation_id"]

for key in (
    "source_document_count",
    "assigned_node_count",
    "unassigned_source_document_count",
    "node_coverage",
    "vector_coverage",
    "label_stage",
):
    print(f"{key:<34} {manifest[key]}")

## Step 3: Browse the hierarchy

Each representative has a level (1 for the broad parent topics, 0 for the narrower child topics) and a `parent_id` linking a child to its parent. A parent-child topic link isn't a business relationship like a farm supplying a lot: it only means the passages are similar.

These topics don't have names, so the cell below shows a few member passage titles for each one.

In [ ]:
generation_filter = f"generation_id eq '{GENERATION}'"
representatives = list(topics_client.search(
    search_text="*",
    filter=f"kind eq 'representative' and {generation_filter}",
    select=["community_id", "parent_id", "level", "member_count", "label"],
    top=1000,
))


def sample_titles(community_id, count=3):
    members = topics_client.search(
        search_text="*",
        filter=f"kind eq 'membership' and community_id eq '{community_id}' and {generation_filter}",
        select=["chunk_id"],
        top=count,
    )
    return [sources_client.get_document(member["chunk_id"], selected_fields=["title"])["title"] for member in members]


for parent in sorted((r for r in representatives if r["level"] == 1), key=lambda r: r["community_id"]):
    print(f"Topic {parent['community_id'][:8]} ({parent['member_count']} passages)")
    for child in sorted((r for r in representatives if r["parent_id"] == parent["community_id"]), key=lambda r: r["community_id"]):
        print(f"  └─ Topic {child['community_id'][:8]} ({child['member_count']} passages): {', '.join(sample_titles(child['community_id']))}")

## Step 4: Pick a passage to start from

Find a passage that mentions Madagascar bean lot MG26-L02 and reuse its stored embedding. This asks "what's near this passage?" rather than embedding a new question, so no embedding model is called.

In [ ]:
seed = next(iter(sources_client.search(
    search_text='"MG26-L02"',
    search_fields=["chunk"],
    select=["chunk_id", "title", "blob_path", "chunk", "text_vector"],
    top=1,
)))
query_vector = seed["text_vector"]
print(f"{seed['title']} ({seed['blob_path']})")
print(textwrap.indent(textwrap.shorten(seed["chunk"], 400), "    "))

## Step 5: Find the nearest passages directly

Run a vector search over all source passages using the seed's embedding. The first result is usually the seed passage itself.

In [ ]:
start = time.perf_counter()
direct = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    select=["chunk_id", "title", "blob_path"],
    top=8,
))
direct_ms = (time.perf_counter() - start) * 1000

for rank, passage in enumerate(direct, 1):
    print(f"{rank}. {passage['title']} ({passage['blob_path']})")

## Step 6: Find the nearest topic, then search within it

Now route the search through a topic:

1. Find the child topic whose centroid is closest to the seed embedding.
2. Look up that topic's member passages.
3. Run the same vector search, filtered to just those members.

The topic narrows the set of candidate passages. That can help focus results, but it can also miss relevant passages in other topics or in the passages that weren't assigned to any topic.

In [ ]:
start = time.perf_counter()
nearest_topic = next(iter(vectors_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="centroid", k_nearest_neighbors=1, exhaustive=True)],
    filter=f"level eq 0 and centroid_method eq 'native-terminal-representative-normalized-float32' and {generation_filter}",
    select=["community_id", "member_count"],
    top=1,
)))
member_ids = [
    member["chunk_id"]
    for member in topics_client.search(
        search_text="*",
        filter=f"kind eq 'membership' and community_id eq '{nearest_topic['community_id']}' and {generation_filter}",
        select=["chunk_id"],
        top=1000,
    )
]
routed = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    filter=f"search.in(chunk_id, '{','.join(member_ids)}', ',')",
    select=["chunk_id", "title", "blob_path"],
    top=8,
))
routed_ms = (time.perf_counter() - start) * 1000

print(f"Nearest topic: {nearest_topic['community_id'][:8]} ({len(member_ids)} member passages)")
print(f"Seed passage is in this topic: {seed['chunk_id'] in member_ids}\n")
for rank, passage in enumerate(routed, 1):
    print(f"{rank}. {passage['title']} ({passage['blob_path']})")

## Step 7: Compare the two approaches

The direct search ran one vector query. The topic-routed search also looked up the nearest topic and its members. The times below are for these lookups only, and don't include embedding a new question or generating an answer.

A passage that appears in only one list wasn't returned by the other search. It's still in the index.

In [ ]:
direct_ids = {passage["chunk_id"] for passage in direct}
routed_ids = {passage["chunk_id"] for passage in routed}

print(f"Direct search:        {len(direct)} passages in {direct_ms:.0f} ms (searched all passages)")
print(f"Topic-routed search:  {len(routed)} passages in {routed_ms:.0f} ms (searched {len(member_ids)} topic members)")
print(f"\nIn both lists:        {len(direct_ids & routed_ids)}")
print(f"Only in direct list:  {len(direct_ids - routed_ids)}")
print(f"Only in topic list:   {len(routed_ids - direct_ids)}")

## Choose the right approach for the question

| Need | Good starting point | Tradeoff |
| --- | --- | --- |
| Ground a direct question in original passages | Standard knowledge base | The model plans queries, picks sources, and writes a cited answer |
| Respond quickly when the app knows which source to use | Microsoft Retriever 1 (MR1) | Faster planning, but the app chooses the source and MR1 returns passages, not an answer |
| Follow explicit relationships across records | Entity graph and GQL | Inspectable joins and provenance, but depends on the quality of the stored relationships |
| Explore groups of related passages | Topics | Broader organization, but topics can be unnamed and don't cover every passage |

Think about the three questions from this lab: the organic seal, the farms behind batch EC26-0811, and the Madagascar bean lot. Which approach would you start with for each, and why?